<a href="https://colab.research.google.com/github/fraycarmona/eml_tabular_grupo_17/blob/ADRIAN/src/EntornosComplejos/estudiosarsa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Estudio SARSA semi-gradiente (Pendulum-v1)

Entrenamiento reproducible con semilla fija `SEED=2024`.

In [ ]:
#@title Copiar el repositorio.

!git clone -b adrian https://github.com/drakopablo/eml_k_bandit_grupo17.git
#!cd eml_k_bandit_grupo17/


In [ ]:
#@title Preparar entorno en Google Colab (opcional)
import os
if 'COLAB_GPU' in os.environ:
    !git clone -b ADRIAN https://github.com/fraycarmona/eml_tabular_grupo_17.git
    %cd /content/eml_tabular_grupo_17/src/EntornosComplejos
    !pip install -q -r requirements.txt


In [ ]:
SEED = 2024
import random
import numpy as np
import torch
import gymnasium as gym

from agents.pendulumsarsaagent import PendulumSarsaAgent
from plotting.plotting import plotlearningcurve, plotlosscurve

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


In [ ]:
env = gym.make('Pendulum-v1')
agent = PendulumSarsaAgent(env=env, seed=SEED)
history_sarsa = agent.train(num_episodes=200, max_steps=200)


In [ ]:
def random_baseline(env_name='Pendulum-v1', episodes=500, max_steps=200, seed=SEED):
    env_r = gym.make(env_name)
    rewards, lengths = [], []
    for ep in range(episodes):
        state, _ = env_r.reset(seed=seed + ep)
        total = 0.0
        for t in range(1, max_steps + 1):
            action = env_r.action_space.sample()
            state, r, terminated, truncated, _ = env_r.step(action)
            total += r
            if terminated or truncated:
                break
        rewards.append(total)
        lengths.append(t)
    env_r.close()
    return {'rewards': rewards, 'lengths': lengths}

baseline = random_baseline()


In [ ]:
plotlearningcurve(
    rewardshistory=history_sarsa['rewards'],
    baselinehistory=baseline['rewards'],
    episode_length_history=history_sarsa['lengths'],
    window=50,
    title='Pendulum-v1: SARSA Semi-gradiente vs Random'
)


In [ ]:
plotlosscurve(
    losshistory=history_sarsa['losses'],
    window=50,
    title='Pendulum-v1: Pérdida TD (SARSA)'
)


In [ ]:
print('Recompensa media final SARSA (100 últimos episodios):', np.mean(history_sarsa['rewards'][-100:]))
print('Recompensa media baseline:', np.mean(baseline['rewards']))
